<a href="https://colab.research.google.com/github/arman-hossain45/Natural_Language_-Processing/blob/main/text_classification_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Here is a simple projecdt about text classification

#TExt classification

In [79]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [80]:
df = pd.read_csv(
    "imbd.csv",
    on_bad_lines='skip',
    encoding='latin-1',
    engine='python'
)


In [81]:
df.head(10)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
5,"Probably my all-time favorite movie, a story o...",positive
6,I sure would like to see a resurrection of a u...,positive
7,"This show was an amazing, fresh & innovative i...",negative
8,Encouraged by the positive comments about this...,negative
9,If you like original gut wrenching laughter yo...,positive


In [82]:
df.shape

(50000, 2)

In [83]:
df.columns

Index(['review', 'sentiment'], dtype='object')

In [84]:
df['sentiment'].value_counts()

,count
sentiment,
positive,25000
negative,25000


In [85]:
df.isna().sum()

,0
review,0
sentiment,0


In [86]:
df.duplicated().sum()

np.int64(418)

In [87]:
df.drop_duplicates(inplace=True)

In [88]:
df.duplicated().sum()

np.int64(0)

# basic preprocessing

In [89]:
import re

# remove html tags

In [90]:
def remove_Html_tags(text):
  cleaned_text=re.sub(re.compile('<.*?>'),"",text)
  return cleaned_text

In [91]:
df['review'] = df['review'].apply(remove_Html_tags)

In [92]:
df['review'][0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.I would say the main appeal of the show is due to the fact that it goes where other shows wo

# Lower case convert

In [93]:
df['review'] = df['review'].apply(lambda x:x.lower())

In [94]:
from nltk.corpus import stopwords

# stop word remove here

In [95]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [96]:
sw_list = stopwords.words('english')
df['review'] = df['review'].apply(lambda x: [item for item in x.split() if item not in sw_list]).apply(lambda x:" ".join(x))

In [97]:
for each_review in df['review']:

    # Sentence → List of words
    words = each_review.split()

    # Stopword remove
    filtered_words = []

    for item in words:
        if item not in sw_list:
            filtered_words.append(item)

    # List → String
    final_review = " ".join(filtered_words)

    # Store back into dataframe
    df['review'] = final_review

In [98]:
df['review'][0]

"one expects star trek movies high art, fans expect movie good best episodes. unfortunately, movie muddled, implausible plot left cringing - far worst nine (so far) movies. even chance watch well known characters interact another movie can't save movie - including goofy scenes kirk, spock mccoy yosemite.i would say movie worth rental, hardly worth watching, however true fan needs see movies, renting movie way see - even cable channels avoid movie."

In [99]:
df.head(4)

,review,sentiment
0,"one expects star trek movies high art, fans ex...",positive
1,"one expects star trek movies high art, fans ex...",positive
2,"one expects star trek movies high art, fans ex...",positive
3,"one expects star trek movies high art, fans ex...",negative


In [100]:
x =df['review']
y=df['sentiment']

In [101]:
y

,sentiment
0,positive
1,positive
2,positive
3,negative
4,positive
...,...
49995,positive
49996,negative
49997,negative
49998,negative


# encode the y means target column

In [102]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
y = encoder.fit_transform(y)

In [103]:
y

array([1, 1, 1, ..., 0, 0, 0])

#train test split

In [104]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size =0.2,random_state=1)

In [105]:
x_train.shape

(39665,)

# now apply bag of words

In [106]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()

In [107]:
x_train_bow = cv.fit_transform(x_train).toarray()
x_test_bow = cv.transform(x_test).toarray()

In [108]:
x_test_bow

array([[1, 1, 1, ..., 2, 1, 1],
       [1, 1, 1, ..., 2, 1, 1],
       [1, 1, 1, ..., 2, 1, 1],
       ...,
       [1, 1, 1, ..., 2, 1, 1],
       [1, 1, 1, ..., 2, 1, 1],
       [1, 1, 1, ..., 2, 1, 1]])

In [109]:
x_test_bow.shape

(9917, 56)

# model declare.

In [110]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score,confusion_matrix

In [111]:
gnb = GaussianNB()
gnb.fit(x_train_bow,y_train)
y_pred = gnb.predict(x_test_bow)


/usr/local/lib/python3.12/dist-packages/sklearn/naive_bayes.py:513: RuntimeWarning: divide by zero encountered in log
  n_ij = -0.5 * np.sum(np.log(2.0 * np.pi * self.var_[i, :]))
/usr/local/lib/python3.12/dist-packages/sklearn/naive_bayes.py:514: RuntimeWarning: invalid value encountered in divide
  n_ij -= 0.5 * np.sum(((X - self.theta_[i, :]) ** 2) / (self.var_[i, :]), 1)


In [112]:
accuracy_score(y_test,y_pred)

0.5075123525259655

In [113]:
confusion_matrix(y_test,y_pred)


array([[5033,    0],
       [4884,    0]])

#Random forest apply

In [114]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100,max_depth=3)
rf.fit(x_train_bow,y_train)
y_pred = rf.predict(x_test_bow)
accuracy_score(y_test,y_pred)

0.4924876474740345

#Apply ngrams

In [115]:
cv_n = CountVectorizer(ngram_range=(1,2))
x_train_bow_n = cv_n.fit_transform(x_train).toarray()
x_test_bow_n = cv_n.transform(x_test).toarray()


In [116]:
rf2 = RandomForestClassifier(n_estimators=100,max_depth=3)
rf2.fit(x_train_bow_n,y_train)
y_pred_n = rf2.predict(x_test_bow_n)
accuracy_score(y_test,y_pred_n)

0.4924876474740345

#Apply TFidf method

In [117]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf = TfidfVectorizer()
x_train_tf = tf.fit_transform(x_train).toarray()
x_test_tf = tf.transform(x_test).toarray()

In [118]:
rf3 = RandomForestClassifier(n_estimators=100,max_depth=3)
rf3.fit(x_train_tf,y_train)
y_pred_tf = rf3.predict(x_test_tf)

In [119]:
accuracy_score(y_test,y_pred_tf)

0.4924876474740345

# Apply word 2 vector

1. pretrain word 2 vec apply criterion is pre train model and my dataset vocabularys similarity is 80 percent equall

2. On my own data set create word 2 veec in details and apply its

In [120]:
!pip install gensim

In [121]:
import gensim

In [122]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

In [123]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [124]:
story = []
for doc in df['review']:
  raw_sent = sent_tokenize(doc)
  for sent in raw_sent:
    story.append(simple_preprocess(sent))

# model which dimension is 100 build in

In [125]:
from gensim.models import Word2Vec

model = Word2Vec(
    window=10,
    min_count=2
)

In [126]:
model.build_vocab(story)

# model trian

In [127]:
model.train(story,total_examples=model.corpus_count,epochs=model.epochs)

(4824838, 16857880)

we create only word vector now we create to reviews vector


In [128]:
len(model.wv.key_to_index)

56

In [129]:
# now we create the reviews vector

In [130]:
def document_vector(doc):
  doc = [word for word in doc.split() if word in model.wv.index_to_key]
  return np.mean(model.wv[doc],axis=0)

# another way to create review vector in details code

In [131]:
# def document_vector(doc):

#     words = doc.split()

#     vectors = []

#     for word in words:
#         if word in model.wv.index_to_key:
#             vectors.append(model.wv[word])

#     if len(vectors) == 0:
#         return np.zeros(model.vector_size)

#     return np.mean(vectors, axis=0)

In [132]:
document_vector(df['review'][0])

array([ 0.12110577, -0.10683344,  0.6067185 ,  0.5344296 ,  0.04545238,
        0.20963773,  0.20353441,  0.12537554,  0.29227862, -0.3670687 ,
       -0.21590164,  0.3975093 ,  0.33107427,  0.04198669, -0.00151466,
       -0.11569384,  0.28827626, -0.21634825, -0.05789457,  0.04557591,
        0.13773467,  0.35484827, -0.31190985,  0.7810395 , -0.71755165,
        0.6008387 , -0.4361814 ,  0.09819137,  0.10224777,  0.1897014 ,
        0.2581058 , -0.06348172, -0.37037742,  0.11091238,  0.19826773,
        0.13304521,  0.18311942, -0.5229213 , -0.12449183,  0.3473055 ,
        0.08532404, -0.25960052,  0.52874064, -0.20728171, -0.1393529 ,
       -0.31162202,  0.09264065, -0.2508056 ,  0.13276698, -0.28524387,
        0.25651574, -0.11692902, -0.5940034 , -0.2900827 , -0.2832664 ,
        0.5009553 ,  0.1547059 ,  0.20221502,  0.0327398 , -0.11546245,
        0.09444349,  0.0539848 ,  0.26981938,  0.5063687 ,  0.166492  ,
        0.05951029, -0.09707481, -0.10196461, -0.6405947 ,  0.07

# now extract all the review and upload it in a space

In [133]:
from tqdm import tqdm

In [134]:
x = []
for doc in tqdm(df['review'].values):
  x.append(document_vector(doc))

100%|██████████| 49582/49582 [00:17<00:00, 2912.26it/s]


In [135]:
x = np.array(x)

In [136]:
x.shape# every reviews here convert it in 100 dimension vector

(49582, 100)

In [137]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
y = encoder.fit_transform(df['sentiment'])

In [137]:
x_train,y_train,x_test,y_test = train_test_split(x,y,test_size=0.2,random_state=1)


In [139]:
x_train_vec = np.array([document_vector(doc) for doc in x_train])

x_test_vec = np.array([document_vector(doc) for doc in x_test])

In [140]:
print(x_train_vec.shape)
print(x_test_vec.shape)

(39665, 100)
(9917, 100)


In [141]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

word2Vec_gnb = GaussianNB()

word2Vec_gnb.fit(x_train_vec, y_train)

y_pred = word2Vec_gnb.predict(x_test_vec)

print(accuracy_score(y_test, y_pred))

0.5075123525259655
